
## Folder 09 — File 02: Incremental Silver Processing:


This notebook takes newly ingested Bronze records, cleans and validates them, and merges valid records into Silver without rebuilding the entire Silver tables.

The key difference from the previous notebook is that we need two separate controls:

- **Auto Loader checkpoint:** tracks which source files have been ingested into Bronze.
- **Silver processing ledger:** tracks which Bronze records have already been processed into Silver.

We'll use a record hash to identify processed Bronze records and Delta MERGE to insert or update Silver records.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# FOLDER 09: INCREMENTAL INGESTION
# FILE 02: INCREMENTAL SILVER PROCESSING
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

VOLUME_SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/"
    f"{VOLUME}/ecommerce_lakehouse"
)

LOG_PATH = f"{BASE_PATH}/logs"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{SILVER_SCHEMA}"
)

dbutils.fs.mkdirs(LOG_PATH)


# ============================================================
# 3. TABLE CONFIGURATION
# ============================================================

DATASET_CONFIG = {

    "customers": {
        "primary_key": ["customer_id"],
        "silver_table": "silver_customers"
    },

    "products": {
        "primary_key": ["product_id"],
        "silver_table": "silver_products"
    },

    "orders": {
        "primary_key": ["order_id"],
        "silver_table": "silver_orders"
    },

    "order_items": {
        "primary_key": ["order_item_id"],
        "silver_table": "silver_order_items"
    },

    "payments": {
        "primary_key": ["payment_id"],
        "silver_table": "silver_payments"
    },

    "shipments": {
        "primary_key": ["shipment_id"],
        "silver_table": "silver_shipments"
    },

    "inventory_snapshots": {
        "primary_key": [
            "product_id",
            "warehouse_id",
            "snapshot_date"
        ],
        "silver_table": "silver_inventory"
    }
}


# ============================================================
# 4. PROCESSING LEDGER CONFIGURATION
# ============================================================

LEDGER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "incremental_silver_processing_ledger"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "incremental_quarantine_records"
)

PROCESSING_REPORT_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}."
    "incremental_silver_processing_report"
)


# Create ledger table if it does not exist.

if not spark.catalog.tableExists(LEDGER_TABLE):

    ledger_schema = """
        dataset_name STRING,
        record_id STRING,
        processed_at TIMESTAMP
    """

    spark.createDataFrame(
        [],
        ledger_schema
    ).write.format("delta").saveAsTable(LEDGER_TABLE)


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def try_cast_column(column_name, target_type):

    return F.expr(
        f"try_cast(`{column_name}` AS {target_type})"
    )


def add_validation_reason(df, dataset_name):

    reasons = []

    # --------------------------------------------------------
    # CUSTOMERS
    # --------------------------------------------------------

    if dataset_name == "customers":

        reasons = [
            (
                F.col("customer_id").isNull()
                | (F.col("customer_id") == ""),
                "Missing customer_id"
            ),
            (
                F.col("created_at").isNull(),
                "Invalid created_at"
            )
        ]

    # --------------------------------------------------------
    # PRODUCTS
    # --------------------------------------------------------

    elif dataset_name == "products":

        reasons = [
            (
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                "Missing product_id"
            ),
            (
                F.col("product_name").isNull()
                | (F.col("product_name") == ""),
                "Missing product_name"
            ),
            (
                F.col("unit_price").isNull()
                | (F.col("unit_price") < 0),
                "Invalid unit_price"
            ),
            (
                F.col("unit_cost").isNull()
                | (F.col("unit_cost") < 0),
                "Invalid unit_cost"
            )
        ]

    # --------------------------------------------------------
    # ORDERS
    # --------------------------------------------------------

    elif dataset_name == "orders":

        reasons = [
            (
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                "Missing order_id"
            ),
            (
                F.col("customer_id").isNull()
                | (F.col("customer_id") == ""),
                "Missing customer_id"
            ),
            (
                F.col("order_timestamp").isNull(),
                "Invalid order_timestamp"
            ),
            (
                F.col("order_status").isNull()
                | (~F.col("order_status").isin(
                    "DELIVERED",
                    "SHIPPED",
                    "PROCESSING",
                    "CANCELLED"
                )),
                "Invalid order_status"
            ),
            (
                F.col("currency").isNull()
                | (F.col("currency") == ""),
                "Missing currency"
            ),
            (
                F.col("order_total").isNull()
                | (F.col("order_total") < 0),
                "Invalid order_total"
            )
        ]

    # --------------------------------------------------------
    # ORDER ITEMS
    # --------------------------------------------------------

    elif dataset_name == "order_items":

        reasons = [
            (
                F.col("order_item_id").isNull()
                | (F.col("order_item_id") == ""),
                "Missing order_item_id"
            ),
            (
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                "Missing order_id"
            ),
            (
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                "Missing product_id"
            ),
            (
                F.col("quantity").isNull()
                | (F.col("quantity") <= 0),
                "Invalid quantity"
            ),
            (
                F.col("unit_price").isNull()
                | (F.col("unit_price") < 0),
                "Invalid unit_price"
            ),
            (
                F.col("discount_rate").isNull()
                | (F.col("discount_rate") < 0)
                | (F.col("discount_rate") > 1),
                "Invalid discount_rate"
            ),
            (
                F.col("line_total").isNull()
                | (F.col("line_total") < 0),
                "Invalid line_total"
            )
        ]

    # --------------------------------------------------------
    # PAYMENTS
    # --------------------------------------------------------

    elif dataset_name == "payments":

        reasons = [
            (
                F.col("payment_id").isNull()
                | (F.col("payment_id") == ""),
                "Missing payment_id"
            ),
            (
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                "Missing order_id"
            ),
            (
                F.col("payment_method").isNull()
                | (F.col("payment_method") == ""),
                "Missing payment_method"
            ),
            (
                F.col("payment_status").isNull()
                | (~F.col("payment_status").isin(
                    "SUCCESS",
                    "PENDING",
                    "FAILED",
                    "REFUNDED"
                )),
                "Invalid payment_status"
            ),
            (
                F.col("amount").isNull()
                | (F.col("amount") < 0),
                "Invalid payment amount"
            ),
            (
                F.col("payment_timestamp").isNull(),
                "Invalid payment_timestamp"
            )
        ]

    # --------------------------------------------------------
    # SHIPMENTS
    # --------------------------------------------------------

    elif dataset_name == "shipments":

        reasons = [
            (
                F.col("shipment_id").isNull()
                | (F.col("shipment_id") == ""),
                "Missing shipment_id"
            ),
            (
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                "Missing order_id"
            ),
            (
                F.col("shipment_status").isNull()
                | (~F.col("shipment_status").isin(
                    "PENDING",
                    "PROCESSING",
                    "SHIPPED",
                    "IN_TRANSIT",
                    "OUT_FOR_DELIVERY",
                    "DELIVERED",
                    "CANCELLED",
                    "RETURNED"
                )),
                "Invalid shipment_status"
            ),
            (
                (
                    F.col("shipment_status") == "DELIVERED"
                )
                & F.col("delivered_date").isNull(),
                "Delivered shipment missing delivered_date"
            ),
            (
                F.col("delivered_date").isNotNull()
                & F.col("shipped_date").isNotNull()
                & (
                    F.col("delivered_date")
                    < F.col("shipped_date")
                ),
                "Delivery date before shipped date"
            )
        ]

    # --------------------------------------------------------
    # INVENTORY
    # --------------------------------------------------------

    elif dataset_name == "inventory_snapshots":

        reasons = [
            (
                F.col("inventory_snapshot_id").isNull()
                | (F.col("inventory_snapshot_id") == ""),
                "Missing inventory_snapshot_id"
            ),
            (
                F.col("product_id").isNull()
                | (F.col("product_id") == ""),
                "Missing product_id"
            ),
            (
                F.col("warehouse_id").isNull()
                | (F.col("warehouse_id") == ""),
                "Missing warehouse_id"
            ),
            (
                F.col("quantity_on_hand").isNull()
                | (F.col("quantity_on_hand") < 0),
                "Invalid quantity_on_hand"
            ),
            (
                F.col("snapshot_date").isNull(),
                "Invalid snapshot_date"
            )
        ]

    # --------------------------------------------------------
    # BUILD REASON COLUMN
    # --------------------------------------------------------

    reason_expressions = [

        F.when(
            condition,
            F.lit(reason)
        )

        for condition, reason in reasons
    ]

    # Rescued records require investigation.
    if "_rescued_data" in df.columns:

        reason_expressions.append(
            F.when(
                F.col("_rescued_data").isNotNull(),
                F.lit("Auto Loader rescued data")
            )
        )

    df = df.withColumn(
        "_quarantine_reason",
        F.concat_ws(
            " | ",
            *reason_expressions
        )
    )

    return df


# ============================================================
# 6. STANDARDIZE DATA TYPES AND VALUES
# ============================================================

def standardize_dataset(df, dataset_name):

    # Trim all string columns.
    for column_name, data_type in df.dtypes:

        if data_type == "string":

            df = df.withColumn(
                column_name,
                F.trim(F.col(column_name))
            )

    # --------------------------------------------------------
    # CUSTOMERS
    # --------------------------------------------------------

    if dataset_name == "customers":

        df = (
            df
            .withColumn(
                "email",
                F.lower(F.col("email"))
            )
            .withColumn(
                "created_at",
                try_cast_column("created_at", "DATE")
            )
        )

    # --------------------------------------------------------
    # PRODUCTS
    # --------------------------------------------------------

    elif dataset_name == "products":

        df = (
            df
            .withColumn(
                "category",
                F.initcap(F.col("category"))
            )
            .withColumn(
                "stock_status",
                F.upper(F.col("stock_status"))
            )
            .withColumn(
                "unit_price",
                try_cast_column("unit_price", "DECIMAL(12,2)")
            )
            .withColumn(
                "unit_cost",
                try_cast_column("unit_cost", "DECIMAL(12,2)")
            )
            .withColumn(
                "created_at",
                try_cast_column("created_at", "DATE")
            )
        )

    # --------------------------------------------------------
    # ORDERS
    # --------------------------------------------------------

    elif dataset_name == "orders":

        df = (
            df
            .withColumn(
                "order_status",
                F.upper(F.col("order_status"))
            )
            .withColumn(
                "currency",
                F.upper(F.col("currency"))
            )
            .withColumn(
                "sales_channel",
                F.initcap(F.col("sales_channel"))
            )
            .withColumn(
                "order_timestamp",
                try_cast_column(
                    "order_timestamp",
                    "TIMESTAMP"
                )
            )
            .withColumn(
                "order_total",
                try_cast_column(
                    "order_total",
                    "DECIMAL(12,2)"
                )
            )
        )

    # --------------------------------------------------------
    # ORDER ITEMS
    # --------------------------------------------------------

    elif dataset_name == "order_items":

        df = (
            df
            .withColumn(
                "quantity",
                try_cast_column("quantity", "INT")
            )
            .withColumn(
                "unit_price",
                try_cast_column(
                    "unit_price",
                    "DECIMAL(12,2)"
                )
            )
            .withColumn(
                "discount_rate",
                try_cast_column(
                    "discount_rate",
                    "DECIMAL(5,4)"
                )
            )
            .withColumn(
                "line_total",
                try_cast_column(
                    "line_total",
                    "DECIMAL(12,2)"
                )
            )
        )

    # --------------------------------------------------------
    # PAYMENTS
    # --------------------------------------------------------

    elif dataset_name == "payments":

        df = (
            df
            .withColumn(
                "payment_method",
                F.upper(F.col("payment_method"))
            )
            .withColumn(
                "payment_status",
                F.upper(F.col("payment_status"))
            )
            .withColumn(
                "amount",
                try_cast_column(
                    "amount",
                    "DECIMAL(12,2)"
                )
            )
            .withColumn(
                "payment_timestamp",
                try_cast_column(
                    "payment_timestamp",
                    "TIMESTAMP"
                )
            )
        )

    # --------------------------------------------------------
    # SHIPMENTS
    # --------------------------------------------------------

    elif dataset_name == "shipments":

        df = (
            df
            .withColumn(
                "carrier",
                F.upper(F.col("carrier"))
            )
            .withColumn(
                "shipment_status",
                F.upper(F.col("shipment_status"))
            )
            .withColumn(
                "shipped_date",
                try_cast_column(
                    "shipped_date",
                    "TIMESTAMP"
                )
            )
            .withColumn(
                "delivered_date",
                try_cast_column(
                    "delivered_date",
                    "TIMESTAMP"
                )
            )
        )

    # --------------------------------------------------------
    # INVENTORY
    # --------------------------------------------------------

    elif dataset_name == "inventory_snapshots":

        df = (
            df
            .withColumn(
                "quantity_on_hand",
                try_cast_column(
                    "quantity_on_hand",
                    "INT"
                )
            )
            .withColumn(
                "snapshot_date",
                try_cast_column(
                    "snapshot_date",
                    "DATE"
                )
            )
            .withColumn(
                "warehouse_id",
                F.upper(F.col("warehouse_id"))
            )
        )

    return df


# ============================================================
# 7. CREATE RECORD IDENTIFIER
# ============================================================

def add_record_identifier(df):

    technical_columns = [
        "_source_file",
        "_source_file_modification_time",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_ingestion_run_id",
        "_rescued_data"
    ]

    business_columns = [
        column_name
        for column_name in df.columns
        if column_name not in technical_columns
        and not column_name.startswith("_")
    ]

    # A stable identifier for each source-file/business-row pair.
    # The source file is included so a later file containing
    # an updated business key can be processed separately.

    df = df.withColumn(
        "_bronze_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(
                    F.col("_source_file"),
                    F.lit("")
                ),
                F.to_json(
                    F.struct(
                        *[
                            F.col(column_name)
                            for column_name in business_columns
                        ]
                    )
                )
            ),
            256
        )
    )

    return df


# ============================================================
# 8. PROCESS ONE DATASET INCREMENTALLY
# ============================================================

def process_incremental_silver(dataset_name):

    print("\n" + "=" * 70)
    print(f"PROCESSING DATASET: {dataset_name}")
    print("=" * 70)

    config = DATASET_CONFIG[dataset_name]

    bronze_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{dataset_name}"
    )

    silver_table = (
        f"{CATALOG}.{SILVER_SCHEMA}."
        f"{config['silver_table']}"
    )

    primary_keys = config["primary_key"]

    # --------------------------------------------------------
    # 8.1 CHECK BRONZE TABLE
    # --------------------------------------------------------

    if not spark.catalog.tableExists(bronze_table):

        print(f"Bronze table not found: {bronze_table}")
        return None

    bronze_df = spark.table(bronze_table)

    # --------------------------------------------------------
    # 8.2 CHECK REQUIRED SOURCE COLUMNS
    # --------------------------------------------------------

    available_columns = set(bronze_df.columns)

    missing_keys = [
        key
        for key in primary_keys
        if key not in available_columns
    ]

    if missing_keys:

        raise ValueError(
            f"Missing business key columns: {missing_keys}"
        )

    required_metadata = [
        "_source_file",
        "_ingestion_timestamp"
    ]

    missing_metadata = [
        column_name
        for column_name in required_metadata
        if column_name not in available_columns
    ]

    if missing_metadata:

        raise ValueError(
            f"Missing Bronze metadata columns: "
            f"{missing_metadata}"
        )

    # --------------------------------------------------------
    # 8.3 CREATE RECORD IDENTIFIER
    # --------------------------------------------------------

    bronze_df = add_record_identifier(bronze_df)

    # --------------------------------------------------------
    # 8.4 REMOVE ALREADY PROCESSED RECORDS
    # --------------------------------------------------------

    processed_ids = (

        spark.table(LEDGER_TABLE)

        .filter(
            F.col("dataset_name") == dataset_name
        )

        .select("record_id")

        .dropDuplicates()

    )

    new_records = (

        bronze_df

        .join(
            processed_ids,
            bronze_df["_bronze_record_id"]
            == processed_ids["record_id"],
            "left_anti"
        )

    )

    new_record_count = new_records.count()

    print(
        f"New Bronze records to process: "
        f"{new_record_count}"
    )

    if new_record_count == 0:

        print("No new Bronze records found.")

        return {
            "dataset_name": dataset_name,
            "new_records": 0,
            "valid_records": 0,
            "quarantined_records": 0,
            "execution_status": "NO_NEW_RECORDS"
        }

    # --------------------------------------------------------
    # 8.5 STANDARDIZE DATA
    # --------------------------------------------------------

    standardized_df = standardize_dataset(
        new_records,
        dataset_name
    )

    # --------------------------------------------------------
    # 8.6 APPLY VALIDATION RULES
    # --------------------------------------------------------

    validated_df = add_validation_reason(
        standardized_df,
        dataset_name
    )

    # --------------------------------------------------------
    # 8.7 SEPARATE VALID AND INVALID RECORDS
    # --------------------------------------------------------

    invalid_df = validated_df.filter(
        F.col("_quarantine_reason") != ""
    )

    valid_df = validated_df.filter(
        F.col("_quarantine_reason") == ""
    )

    invalid_count = invalid_df.count()
    valid_count = valid_df.count()

    print(f"Valid records: {valid_count}")
    print(f"Invalid records: {invalid_count}")

    # --------------------------------------------------------
    # 8.8 PREPARE QUARANTINE RECORDS
    # --------------------------------------------------------

    if invalid_count > 0:

        business_columns = [
            column_name
            for column_name in invalid_df.columns
            if column_name not in [
                "_quarantine_reason",
                "_bronze_record_id"
            ]
        ]

        quarantine_df = (

            invalid_df

            .select(
                F.lit(dataset_name).alias("dataset_name"),

                F.col("_bronze_record_id").alias("record_id"),

                F.col("_quarantine_reason").alias(
                    "quarantine_reason"
                ),

                F.to_json(
                    F.struct(
                        *[
                            F.col(column_name)
                            for column_name in business_columns
                        ]
                    )
                ).alias("record_data"),

                F.current_timestamp().alias("quarantined_at")
            )

        )

        if not spark.catalog.tableExists(QUARANTINE_TABLE):

            quarantine_df.write.format(
                "delta"
            ).saveAsTable(
                QUARANTINE_TABLE
            )

        else:

            quarantine_delta = DeltaTable.forName(
                spark,
                QUARANTINE_TABLE
            )

            (

                quarantine_delta.alias("target")

                .merge(
                    quarantine_df.alias("source"),
                    """
                    target.dataset_name = source.dataset_name
                    AND target.record_id = source.record_id
                    """
                )

                .whenNotMatchedInsertAll()

                .execute()

            )

    # --------------------------------------------------------
    # 8.9 DEDUPLICATE VALID RECORDS
    # --------------------------------------------------------

    if valid_count > 0:

        key_condition = reduce(
            lambda left, right: left & right,
            [
                F.col(key).isNotNull()
                & (F.col(key).cast("string") != "")
                for key in primary_keys
            ]
        )

        valid_df = valid_df.filter(key_condition)

        window_spec = (

            Window

            .partitionBy(*primary_keys)

            .orderBy(
                F.col("_ingestion_timestamp").desc_nulls_last(),
                F.col("_source_file").desc_nulls_last()
            )

        )

        valid_df = (

            valid_df

            .withColumn(
                "_row_number",
                F.row_number().over(window_spec)
            )

            .filter(
                F.col("_row_number") == 1
            )

            .drop("_row_number")

        )

        # ----------------------------------------------------
        # 8.10 REMOVE INTERNAL VALIDATION COLUMN
        # ----------------------------------------------------

        valid_df = valid_df.drop(
            "_quarantine_reason"
        )

        # ----------------------------------------------------
        # 8.11 MERGE INTO SILVER
        # ----------------------------------------------------

        if not spark.catalog.tableExists(silver_table):

            (

                valid_df

                .write

                .format("delta")

                .mode("overwrite")

                .saveAsTable(silver_table)

            )

            print(
                f"Created Silver table: {silver_table}"
            )

        else:

            target_delta = DeltaTable.forName(
                spark,
                silver_table
            )

            merge_condition = " AND ".join(
                [
                    f"target.`{key}` <=> source.`{key}`"
                    for key in primary_keys
                ]
            )

            (

                target_delta.alias("target")

                .merge(
                    valid_df.alias("source"),
                    merge_condition
                )

                .whenMatchedUpdateAll(
                    condition="""
                    source._ingestion_timestamp
                    >= target._ingestion_timestamp
                    """
                )

                .whenNotMatchedInsertAll()

                .execute()

            )

        print(
            f"Silver MERGE completed: {silver_table}"
        )

    # --------------------------------------------------------
    # 8.12 UPDATE PROCESSING LEDGER
    # --------------------------------------------------------

    # Mark both valid and quarantined records as processed.
    # This prevents invalid records from being repeatedly
    # evaluated on every notebook run.

    processed_records = (

        new_records

        .select(
            F.lit(dataset_name).alias("dataset_name"),

            F.col("_bronze_record_id").alias("record_id")
        )

        .dropDuplicates(
            ["dataset_name", "record_id"]
        )

        .withColumn(
            "processed_at",
            F.current_timestamp()
        )

    )

    ledger_delta = DeltaTable.forName(
        spark,
        LEDGER_TABLE
    )

    (

        ledger_delta.alias("target")

        .merge(
            processed_records.alias("source"),
            """
            target.dataset_name = source.dataset_name
            AND target.record_id = source.record_id
            """
        )

        .whenNotMatchedInsertAll()

        .execute()

    )

    # --------------------------------------------------------
    # 8.13 RETURN PROCESSING METRICS
    # --------------------------------------------------------

    result = {

        "dataset_name": dataset_name,

        "new_records": new_record_count,

        "valid_records": valid_count,

        "quarantined_records": invalid_count,

        "execution_status": "SUCCESS"

    }

    print(result)

    return result


# ============================================================
# 9. PROCESS ALL DATASETS
# ============================================================

processing_results = []

for dataset_name in DATASET_CONFIG.keys():

    try:

        result = process_incremental_silver(
            dataset_name
        )

        if result is not None:

            processing_results.append(result)

    except Exception as error:

        print(
            f"FAILED: {dataset_name}: {str(error)}"
        )

        processing_results.append({

            "dataset_name": dataset_name,

            "new_records": 0,

            "valid_records": 0,

            "quarantined_records": 0,

            "execution_status": "FAILED"

        })


# ============================================================
# 10. CREATE PROCESSING REPORT
# ============================================================

if processing_results:

    report_df = spark.createDataFrame(
        processing_results
    )

    report_df = report_df.withColumn(
        "execution_timestamp",
        F.current_timestamp()
    )

    if spark.catalog.tableExists(
        PROCESSING_REPORT_TABLE
    ):

        (

            report_df

            .write

            .format("delta")

            .mode("append")

            .saveAsTable(PROCESSING_REPORT_TABLE)

        )

    else:

        (

            report_df

            .write

            .format("delta")

            .saveAsTable(PROCESSING_REPORT_TABLE)

        )

    display(
        report_df.orderBy("dataset_name")
    )


# ============================================================
# 11. DISPLAY SILVER TABLE COUNTS
# ============================================================

for dataset_name, config in DATASET_CONFIG.items():

    silver_table = (
        f"{CATALOG}.{SILVER_SCHEMA}."
        f"{config['silver_table']}"
    )

    if spark.catalog.tableExists(silver_table):

        count = spark.table(
            silver_table
        ).count()

        print(
            f"{silver_table}: {count} records"
        )

    else:

        print(
            f"Silver table missing: {silver_table}"
        )


# ============================================================
# 12. FINAL PROCESSING STATUS
# ============================================================

print("\n" + "=" * 70)
print("INCREMENTAL SILVER PROCESSING COMPLETED")
print("=" * 70)

print(
    f"Processing report: {PROCESSING_REPORT_TABLE}"
)

print(
    f"Processing ledger: {LEDGER_TABLE}"
)

print(
    f"Quarantine table: {QUARANTINE_TABLE}"
)



## Verify the output:


### Run these SQL queries after the notebook finishes.


### A. Check processing report

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_silver.incremental_silver_processing_report
ORDER BY execution_timestamp DESC;


### B. Check processing ledger

In [0]:
%sql

SELECT
    dataset_name,
    COUNT(*) AS processed_record_count
FROM workspace.ecommerce_silver.incremental_silver_processing_ledger
GROUP BY dataset_name
ORDER BY dataset_name;


### C. Check quarantined records

In [0]:
%sql

SELECT
    dataset_name,
    quarantine_reason,
    COUNT(*) AS invalid_record_count
FROM workspace.ecommerce_silver.incremental_quarantine_records
GROUP BY dataset_name, quarantine_reason
ORDER BY invalid_record_count DESC;


### D. Check Silver tables

In [0]:
%sql

SELECT COUNT(*) AS total_customers
FROM workspace.ecommerce_silver.silver_customers;

SELECT COUNT(*) AS total_products
FROM workspace.ecommerce_silver.silver_products;

SELECT COUNT(*) AS total_orders
FROM workspace.ecommerce_silver.silver_orders;


## Processing Architecture:


### Bronze to Silver Incremental Processing Flow:



```text
┌────────────────────────────────────────────────────┐
│                 BRONZE DELTA TABLES                │
│                                                    │
│              Newly Ingested Records                │
└────────────────────────────────────────────────────┘
                          │
                          ▼
┌────────────────────────────────────────────────────┐
│                 PROCESSING LEDGER CHECK             │
│                                                    │
│  • Generate Record Hashes                          │
│  • Check Previously Processed Hashes               │
│  • Exclude Previously Processed Records            │
│  • Identify New or Unprocessed Records             │
└────────────────────────────────────────────────────┘
                          │
                          ▼
┌────────────────────────────────────────────────────┐
│             STANDARDIZATION AND VALIDATION         │
│                                                    │
│  • Trim Whitespace                                │
│  • Cast Data Types                                │
│  • Normalize Values                               │
│  • Apply Data Quality Rules                       │
└────────────────────────────────────────────────────┘
                          │
                 ┌────────┴────────┐
                 ▼                 ▼
┌────────────────────────┐  ┌────────────────────────┐
│     VALID RECORDS      │  │    INVALID RECORDS    │
│                        │  │                        │
│  • Passed Validation   │  │  • Failed Validation  │
│  • Ready for MERGE     │  │  • Failure Reasons    │
└────────────────────────┘  └────────────────────────┘
                 │                 │
                 ▼                 ▼
┌────────────────────────┐  ┌────────────────────────┐
│    DELTA LAKE MERGE    │  │    QUARANTINE TABLE    │
│                        │  │                        │
│  • Update Existing     │  │  • Store Invalid Data │
│    Records             │  │  • Store Failure      │
│  • Insert New Records  │  │    Reasons            │
└────────────────────────┘  └────────────────────────┘
                 │                 │
                 └────────┬────────┘
                          ▼
┌────────────────────────────────────────────────────┐
│                PROCESSING LEDGER UPDATED            │
│                                                    │
│  • Record Successfully Processed Hashes            │
│  • Maintain Processing Status                     │
│  • Track Processing History                       │
└────────────────────────────────────────────────────┘
```


### Processing Components


| Component | Responsibility |
|---|---|
| Bronze Delta Tables | Provide newly ingested raw records. |
| Processing Ledger | Tracks previously processed record hashes. |
| Record Hashing | Generates fingerprints to identify records. |
| Standardization | Trims, casts, and normalizes incoming data. |
| Validation | Applies data quality and business rules. |
| Delta MERGE | Updates matching Silver records and inserts new records. |
| Quarantine | Stores invalid records and their failure reasons. |
| Ledger Update | Records processing outcomes for future executions. |